In [14]:
import numpy as np
import glob
import healpy as hp
import time

from vorothreshold import voronoi_threshold 
from vorothreshold.read_funcs import read_voronoi_vide, read_adjfile
from vorothreshold.utilities import from_XYZ_to_rRAdec, from_rRAdec_to_XYZ, ComovingDistanceOverh, RedshiftFromComovingDistanceOverh, StrHminSec
from vorothreshold.masks import borders_mask_bruteforce, dist_limit_mask, borders_mask
from vorothreshold.overlaps import overlapping_fraction, select_overlaps, compute_max_dist2, \
    order_ids_tracers_selected_in_voxels, order_coord_tracers_in_voxels_ids_rev_copy, nearest_cell_core, nearest_cell_loop

import matplotlib.pyplot as plt

In [2]:
vide_path='data/lightcone/examples/example_observation/sample_example_observation/'
OmegaM = 0.3
sample_name = 'example_observation'
ids_voro, VoroVol, VoroXYZ, RA, Dec, redshift = read_voronoi_vide(vide_path,sample_name)

In [3]:
adjfile = glob.glob(vide_path+'/adj_*')[0] #vide_path + '/adj_' + vide_out_name + '.dat'
neighbor_ptr, neighbor_ids = read_adjfile(adjfile)

# recover vide_out_name
vide_out_name = adjfile.split('adj_')[1].split('.dat')[0]
#if ID_core is None:
# Load ids of cells belonging to minima
ID_core = np.loadtxt(vide_path+'/untrimmed_voidDesc_all_'+vide_out_name+'.out', comments='#', skiprows=2)[:,2].astype(np.int_)
# Load Voronoi cells volume, ids and tracers position
ids_voro, VoroVol, VoroXYZ, RAvoro, DECvoro, redshift_voro = read_voronoi_vide(vide_path,vide_out_name)

#OmegaM = load_pickle_safe(vide_path+'/sample_info.dat')['omegaM']


dist_z = ComovingDistanceOverh(OmegaM,-1.,0.)

dist_voro = dist_z.get_dist(redshift_voro)
VoroXYZ[:,:] = np.array(from_rRAdec_to_XYZ(dist_voro,RAvoro,DECvoro)).T

max_num_part = int(5 * np.max(np.loadtxt(vide_path+'/untrimmed_centers_all_'+vide_out_name+'.out', comments="#")[:,9]))


hist_z, z_bins = np.histogram(redshift,bins=31)


mask_gal_file = vide_path + 'mask_map.fits'
mask_pix = hp.read_map(mask_gal_file)
sky_frac = np.sum(mask_pix)/mask_pix.shape[0]

comov_bins = dist_z.get_dist(z_bins)
comov3 = dist_z.get_dist(z_bins)**3
shell_vol = 4*np.pi/3.*sky_frac*(comov3[1:] - comov3[:-1])
z_mean = 0.5 * (z_bins[1:] + z_bins[:-1])
dist_mean = dist_z.get_dist(z_mean)
mean_dens = np.mean((hist_z/shell_vol)[(dist_mean>=75) & (dist_mean<=400)])

tracer_dens = np.full(redshift.shape[0],mean_dens)


In [4]:
threshold = [0.3]
void_selected, ID_voro_dict, Xcm, Vol_interp, Ncells_in_void, ell_eigenvalues, ell_eigenvectors = voronoi_threshold(
    threshold,ID_core,neighbor_ptr,neighbor_ids,VoroXYZ,VoroVol,tracer_dens,Lbox=-1,nthreads=-1,verbose=True,max_num_part=max_num_part)
        


    voronoi_threshold started. 

    nthreads set to 32

    max_num_part set to 3685

    computation started
    done, 0 h 0 min 11.449962854385376 sec. 



In [5]:

i_min = np.argmin(dist_voro)
i_max = np.argmax(dist_voro)
comov_range_base = [250.,400.]

comov_range = np.array(comov_range_base)
if len(comov_range.shape) == 1:
    comov_range = np.empty((len(threshold),2))
    comov_range[:,0] = min(comov_range_base)
    comov_range[:,1] = max(comov_range_base)
elif comov_range.shape[0] < len(threshold):
    comov_range = np.empty((len(threshold),2))
    comov_range[:,0] = min(comov_range_base)
    comov_range[:,1] = max(comov_range_base)
print(comov_range)

[[250. 400.]]


In [6]:
nside = hp.get_nside(mask_pix)
healpix_mask = dict()
ids_selected = dict()
for ith in range(len(threshold)):
    trs_mask = (dist_voro >= comov_range[ith,0]) & (dist_voro <= comov_range[ith,1])
    ang_paddig_rad = 2. * np.max(tracer_dens[trs_mask]**(-1./3.) / dist_voro[trs_mask])
    npadding_ang = int((ang_paddig_rad + hp.nside2resol(nside)) / hp.nside2resol(nside))
    mask_ids, mask_voro, healpix_mask[ith] = borders_mask(mask_pix,RAvoro,DECvoro,ID_voro_dict,Ncells_in_void[:,ith],npadding_ang)
    ids_selected[ith] = dist_limit_mask(mask_ids,Xcm[:,ith,:],comov_range[ith,0],comov_range[ith,1],
                            VoroXYZ,Ncells_in_void[:,ith],ID_voro_dict) 
    print(ang_paddig_rad,npadding_ang,ids_selected[ith].shape)
                


0.059876929758343425 8 (136,)


In [7]:
ids_ovlp_frac = dict()
Vol_ovlp_frac = dict()
num_ovlps_frac = dict()
sort_by_vol_frac = dict()
id_out_frac = dict()

In [8]:
frac_ovlp = 0.3

id_out_frac[ith] = dict()
ids_ovlp_frac[ith], Vol_ovlp, Vol_ovlp_frac[ith], num_ovlps_frac[ith] = overlapping_fraction(
    Xcm[:,ith,:], Vol_interp[:,ith], Ncells_in_void[:,ith], VoroXYZ, VoroVol, ID_voro_dict,
    Lbox=-1,lightcone=True,id_selected=ids_selected[ith],nthreads=-1,verbose=True)
sort_by_vol_frac[ith] = np.argsort(Vol_interp[ids_selected[ith],ith])[::-1].astype(dtype=np.int_,order='C')

id_out_frac[ith][frac_ovlp] = select_overlaps(frac_ovlp,ids_selected[ith],sort_by_vol_frac[ith], ids_ovlp_frac[ith], Vol_ovlp_frac[ith], num_ovlps_frac[ith])


overlapping_fraction started.

    nthreads set to 32

    R_max computed. Max val = 44.66014755275007

    Lbox not passed, using xyz_vds as reference:
    min(xyz_vds) = -292.17148326489206 -370.03584395371087 24.49048957489266
    max(xyz_vds) = 276.63418947328694 -102.15536361583469 320.68229381827257
    Lbox = 568.805672738179

    ngrid not passed. Set to optimal value: 13

    order_ids_tracers_selected_in_voxels started
    done, 0 h 0 min 0.41770505905151367 sec.

    computation started (periodic-boundaries condition off)
    done, 0 h 0 min 3.9503517150878906 sec. 



In [9]:
print(id_out_frac[ith][frac_ovlp].shape,id_out_frac[ith][frac_ovlp])

(122,) [  7  66  70   0  28  62  77  41  48   8  45 133  18 108  86  38  47  51
 118  13 102 124  88  10  42  30 109   9  65  21  74  39  82  33  72  36
  26 115  23  25  92  71  87 120 113 129 110  56  32 125 126  93  49  96
  54  94  24  79 121  73  27 134 114  46  20  90  37  64  75  59  19 117
  34   2   4  67  89 127 112  95  29  69 101  55  11 135  81  16 104 119
   1  63  53  43 122  80   3 132  12  60  14  15 123   5  17  52  84  61
 111  22 105 100 130  76   6  58  91  50 116  35  68  31]


In [10]:
id_ovlp_out = ids_selected[ith][id_out_frac[ith][frac_ovlp]]
print(id_ovlp_out.shape,id_ovlp_out)

(122,) [ 15 113 124   0  54 108 132  68  80  16  72 211  37 174 147  65  79  86
 188  24 168 196 150  19  69  57 175  18 112  42 129  66 140  60 127  63
  49 185  44  48 157 126 149 191 181 203 176  98  59 198 199 158  84 161
  92 159  45 135 192 128  52 212 182  76  41 154  64 110 130 103  38 187
  61   7   9 118 151 200 178 160  55 123 167  97  21 213 138  33 170 190
   4 109  91  70 193 136   8 209  22 104  30  32 195  11  34  89 142 106
 177  43 171 166 204 131  14 102 155  85 186  62 120  58]


start checking center in void

In [11]:
Ncells = np.copy(Ncells_in_void[:,ith])
id_selected = np.arange(Ncells.shape[0])[Ncells > 1.]
xyz_vds = np.copy(Xcm[:,ith,:])
max_dist_vds = compute_max_dist2(Ncells,xyz_vds,VoroXYZ,id_selected,ID_voro_dict)**0.5
print('R_max:',np.max(max_dist_vds),max_dist_vds.shape)


offset = np.min(VoroXYZ,axis=0)
max_values = np.max(VoroXYZ,axis=0)
Lbox = np.max(max_values - offset)

offset -= Lbox*1e-4
max_values += Lbox*1e-4
Lbox = np.max(max_values - offset)
ngrid = max(int(round(Lbox / 20.)),4)


print("    min(xyz_vds) =",*offset,flush=True)
print("    max(xyz_vds) =",*max_values,flush=True)
print("    Lbox =",Lbox,flush=True)
print("    ngrid:",ngrid,flush=True)


voxel_side = Lbox / ngrid

print('\n    order_ids_tracers_selected_in_voxels started',flush=True)
#print(VoroXYZ.shape,VoroXYZ[:10,:])

t0 = time.time()

#IDs_vds_ordered, voxel_ptr = order_ids_tracers_in_voxels(xyz_vds, ngrid, Lbox)
xyz_trs_out, ids_reverse, ind_vox = order_coord_tracers_in_voxels_ids_rev_copy(VoroXYZ - offset, ngrid, Lbox)

dt = time.time() - t0
print("    done,",StrHminSec(dt),flush=True)

#print(np.min(xyz_trs_out),np.max(xyz_trs_out),xyz_trs_out[:10,:])

R_max: 99.34885906394673 (214,)
    min(xyz_vds) = -404.89752 -433.31003 -84.366234
    max(xyz_vds) = 379.357 432.73975 402.80176
    Lbox = 866.0498
    ngrid: 43

    order_ids_tracers_selected_in_voxels started
    done, 0 h 0 min 0.4385802745819092 sec.


In [12]:
print(ids_reverse)

[91561 96469  1078 ... 21302 21303 21305]


In [18]:
id_loop = 87
R_max = np.max(max_dist_vds)
Rvds = Vol_interp**(1/3)*3/(4*np.pi)

max_iterations = 3
t0 = time.time()
ids_closest = nearest_cell_loop(
    id_selected, xyz_vds-offset, xyz_trs_out, ids_reverse, ind_vox, ngrid, voxel_side, max_iterations)
print(time.time()-t0)

0.019082307815551758


In [17]:
dist_closest = np.sum((xyz_vds[id_selected,:] - VoroXYZ[ids_closest,:])**2,axis=1)**0.5
dist_test = np.zeros(id_selected.shape[0])

t0 = time.time()
for ii in range(id_selected.shape[0]):
    dist_test[ii] = np.min(np.sum((xyz_vds[ii,:] - VoroXYZ)**2,axis=1)**0.5)
print(time.time()-t0)

print(np.max(dist_closest -dist_test))

0.30156803131103516
0.0
